# 🔄 Agent-to-Agent: Hierarchical Research with Bigdata.com

This notebook shows a **hierarchical agent** that checks internal data first, then escalates to the **Bigdata.com Research Agent** for deep, cited research when needed.

## What This Demonstrates

This notebook delivers a cited, multi-source answer in one flow—combining internal portfolios and research with the **Bigdata.com Research Agent** for deep external analysis when needed. **One research tool keeps your agent capable of comprehensive, sourced answers** without you managing multiple search APIs. The result: faster answers from internal data when possible, and traceable insights with inline source links when you escalate to Research.

**Internal-first flow:**
- **Primary Agent** queries internal DB (portfolios, holdings, transactions) and internal research (FAISS vector store).
- **Escalation** to Bigdata.com Research Agent only when the question needs external, multi-source analysis (20–60s, full citations).

**Research Agent:**
- Multi-step reasoning with RAG; answers include inline citations as source-name hyperlinks (no citation numbers).
- Retry, stream timeout, and logging (production-ready; see `research_client.py`).

**Framework flexibility:**
> This demo uses **LangChain** and **LangSmith**. The pattern—internal tools first, then one “research” tool—works with **CrewAI**, **AutoGen**, or custom graphs. The key is tool ordering and system prompt that prefers internal sources.

## Architecture

![Agent to Bigdata Research Agent](./static/agent-research.png)

**Key benefits:** Cost and latency control (internal answers are fast); full citations when escalating; reusable `langgraph_core` setup and tools.

## Use Cases

| Role | Example Questions |
|------|-------------------|
| **Equity Research** | Thesis validation, competitive analysis |
| **Credit Research** | Covenant analysis, refinancing risks |
| **Credit Risk** | Counterparty exposure, default drivers |

---

## Langsmith Tracing

![LangSmith Tracing](./static/langsmith.png)

## 1️⃣ Install Dependencies

> **Note:** If you're using `uv` to manage dependencies (recommended), you can skip the pip install cell below. Dependencies are already installed via `uv pip install -r requirements.txt`.

In [7]:
# Dependencies are installed via uv: uv pip install -r requirements.txt
%pip install langchain langchain-openai langchain-community faiss-cpu requests python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 2️⃣ Import Libraries

Import LangChain, display helpers, and reusable setup/tools from `langgraph_core`. Display helpers: `display_query`, `display_tools_used`, `display_response`, `display_citations`.

In [8]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

# LangChain
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent as langchain_create_agent

# Reusable core: environment, data sources, tools, display
import sys
sys.path.append(".")
from langgraph_core import (
    setup_environment,
    create_financial_database,
    create_vector_store,
    get_database_tools,
    get_vectorstore_tools,
    get_bigdata_tools,
    get_research_agent_tool,
    run_agent_query,
    display_query,
    display_response,
    display_tools_used,
    display_citations,
)
load_dotenv()
print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 3️⃣ Setup Environment

In [9]:
# Setup with LangSmith tracing
config = setup_environment(
    langsmith_project='bigdata-agent-to-agent',
    enable_tracing=True
)
print("\n🔗 View agent traces: https://smith.langchain.com")

✅ LangSmith tracing enabled → Project: bigdata-agent-to-agent
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...

🔗 View agent traces: https://smith.langchain.com


## 4️⃣ Initialize Data Sources

Create the internal database and vector store with sample financial data.

In [10]:
# Create SQLite database with portfolios, holdings, transactions
create_financial_database()
# Create vector store with internal research documents
create_vector_store()
print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")

✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)


## 5️⃣ Create Hierarchical Agent

This agent is configured to:
1. Check **internal sources first** (faster, proprietary data)
2. Use **quick external lookups** for company info
3. **Escalate** to Research Agent only when needed

System prompt and agent are defined below so the flow is transparent.

**Note:** This should to be updated based on need.

In [11]:
# System prompt: internal-first, then escalate to Research Agent; inline citations as hyperlinks
HIERARCHICAL_SYSTEM_PROMPT = """You are a senior financial research analyst with access to both internal company systems and the Bigdata.com Research Agent for external research.

**RESEARCH HIERARCHY (follow in order):**

1. **ALWAYS check internal sources FIRST:**
   - `internal_query_database` - Portfolio positions, transactions, and account data
   - `internal_portfolio_summary` - Quick overview of specific portfolios
   - `internal_search_research` - Internal research documents, investment theses, and analyst notes

2. **For entity resolution (as needed):**
   - `bigdata_lookup_company` - Get Bigdata entity IDs for company identification

3. **ESCALATE to Research Agent ONLY when internal sources are insufficient:**
   - `bigdata_research_agent` - For deep analysis, market trends, current events, or external information

**CRITICAL: SINGLE RESEARCH AGENT CALL RULE**
- Before calling the Research Agent, STOP and consolidate ALL external research needs into ONE comprehensive question
- Review the user's full request and identify every piece of external information needed
- Combine all needs into a single, well-structured query covering all aspects
- Research Agent calls take few minutes, each—multiple calls waste time and resources
- If you realize you need additional external information after a Research Agent call, incorporate it into your response using only what you have OR clearly explain what additional research would require

**Query Consolidation Example:**
❌ BAD: Three separate calls for "NVDA earnings", "semiconductor sector outlook", "AI chip competition"
✅ GOOD: Single call: "Provide analysis of NVIDIA including: (1) recent earnings performance and guidance, (2) semiconductor sector outlook, (3) competitive positioning in AI chips vs AMD and Intel"

**Decision Framework:**
| Question Type | Action |
|--------------|--------|
| Our portfolio/holdings | Internal DB first |
| Company we hold | Internal research first → escalate if gaps |
| Company we don't hold | Research Agent (single comprehensive call) |
| Market trends / sector analysis | Research Agent (single comprehensive call) |
| Current news / events | Research Agent (single comprehensive call) |
| Mixed (internal + external needs) | Internal first → consolidate all external gaps → single Research Agent call |

**Citation Format:**
- Research Agent output: Inline hyperlinks only—[Source Name](url). No citation numbers [1], [2]. No "Sources" section at end.
- Internal data: Brief attribution like "From internal database" or "According to internal research."

**Available Portfolios:** PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders).

**Response Guidelines:**
- Do not offer follow-up question suggestions
- If the Research Agent response doesn't fully answer the query, work with available information rather than making additional calls
"""

# Build tools: internal first, then lookup, then research agent
tools = (
    get_database_tools() +
    get_vectorstore_tools() +
    [t for t in get_bigdata_tools() if t.name == "bigdata_lookup_company"] +
    get_research_agent_tool()
)
llm = ChatOpenAI(model="gpt-5", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
agent = langchain_create_agent(llm, tools, system_prompt=HIERARCHICAL_SYSTEM_PROMPT)
tool_names = [t.name for t in tools]
print(f"✅ Hierarchical agent created with {len(tools)} tools:")
print(f"   Internal: {[n for n in tool_names if n.startswith('internal_')]}")
print(f"   External: {[n for n in tool_names if n.startswith('bigdata_')]}")

✅ Hierarchical agent created with 5 tools:
   Internal: ['internal_query_database', 'internal_portfolio_summary', 'internal_search_research']
   External: ['bigdata_lookup_company', 'bigdata_research_agent']


---

# 📈 Equity Research Use Cases

Questions that equity analysts typically ask.

### Query 1: Investment Thesis Validation (Internal + External)

Combines internal research with market data - may escalate for current news.

In [13]:
query = """
Review our investment thesis for NVIDIA:
1. What does our internal research say about NVIDIA's competitive position?
2. What recent market developments might affect this thesis?
3. Should we adjust our position based on current information?
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

🔧 internal_search_research: {
  "query": "NVIDIA competitive position, AI accelerators, CUDA moat, software ecosystem, supply chain, risks, internal thesis",
  "top_k": 5
}...
🔧 internal_query_database: {
  "sql_query": "SELECT portfolio_id, ticker, company_name, shares, avg_cost, current_price, market_value, unrealized_pnl FROM holdings WHERE ticker ...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF002"
}...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF003"
}...
🔧 bigdata_research_agent: {
  "query": "Provide a concise, up-to-date analysis of NVIDIA (NVDA) as of late January 2026 to inform a portfolio review. Please cover:\n\n1) Recent...


Here’s a concise, portfolio‑actionable review based on our internal research, our current positions, and recent market developments.

1) Our internal thesis on NVIDIA’s competitive position
- Leadership in AI accelerators: NVIDIA remains the de facto standard for AI training and high‑end inference, with the Hopper family (H100/H200) still capacity‑constrained and Blackwell (B100/B200) expected to extend performance leadership. According to internal research, Blackwell targets ~2.5x perf uplift vs Hopper, reinforcing the hardware lead.
- Durable software moat: CUDA’s large developer base (4M+ per internal estimates) and the broader NVIDIA AI Enterprise stack create high switching costs. Internal analysis repeatedly cites CUDA/SDKs/tools as the key barrier vs AMD/Intel.
- Expanding TAM to inference: Internal thesis flags a large, under‑penetrated inference opportunity as enterprises deploy AI at scale (internal estimate ~$150B TAM by 2027), benefiting accelerators, networking, and software.
- Key risks we track: China export restrictions (internal risk memo notes 20–25% revenue exposure at risk), supply chain bottlenecks, and intensifying AMD competition. Valuation sensitivity and AI spending cyclicality are also flagged in our Technology Sector Risk Assessment. 
- Prior internal stance: Strategy memo (Jan 2025) recommended increasing NVDA weight given demand > supply, while also highlighting the need to monitor China policy and valuation risks.

2) Recent market developments that may affect the thesis
Supportive
- Demand and margins remain strong: Management has guided to very robust near‑term revenue with non‑GAAP gross margin around the mid‑70% range as supply remains tight and accelerators are “sold out,” reflecting persistent hyperscaler demand [Yahoo! Finance](https://finance.yahoo.com/news/ai-leader-nvidia-forecasts-fourth-212147945.html), [Nasdaq](https://www.nasdaq.com/articles/buy-3-ai-semiconductor-powerhouses-poised-dominate-2026), [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-breaks-expectations-massive-earnings-114938956.html), [Nasdaq](https://www.nasdaq.com/articles/ai-bubble-worries-are-rising-nvidias-319-billion-profit-says-otherwise).
- Hyperscaler AI capex still rising: Microsoft recently highlighted record capex (roughly half on chips), underscoring sustained AI infrastructure build‑outs that map to accelerator demand [Yahoo! Finance](https://finance.yahoo.com/news/ai-leader-nvidia-forecasts-fourth-212147945.html).
- Upcoming catalysts: Next earnings (late Feb) and continued Blackwell ramp are key near‑term events; expectations are elevated [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-eyes-openai-investment-buy-182748007.html).

Headwinds/Watch items
- China export controls: Sales of China‑specific data center SKUs remain limited, highlighting a persistent headwind relative to pre‑restriction levels [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-breaks-expectations-massive-earnings-114938956.html). This aligns with our internal risk view on China exposure.
- Competitive pressure vs margins: As rivals offer cheaper alternatives, any pricing pressure could show up in gross margin—an area investors are watching closely [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-4-trillion-stock-rally-104747765.html).
- Valuation sensitivity: Our internal risk memo notes the risk of multiple compression for “AI leaders” if AI monetization lags expectations.

Note: Internal AMD research continues to show MI300 traction (Azure/Oracle deployments) but also notes ROCm still trails CUDA; we see AMD as most relevant near‑term competitor in certain inference/training workloads (according to internal research).

3) Positioning recommendations (based on current info)
Context from internal database:
- PF002 (AI & Semiconductor Focus): NVDA $10.51m of $15.99m total MV ≈ 65.7% weight (From internal database).
- PF003 (Diversified Tech Leaders): NVDA $7.00m of $23.23m total MV ≈ 30.1% weight (From internal database).

Our take
- Fundamental backdrop remains favorable near term: Demand visibility is strong into the Blackwell cycle; margins remain elevated; hyperscaler capex is supportive [Yahoo! Finance](https://finance.yahoo.com/news/ai-leader-nvidia-forecasts-fourth-212147945.html). This is directionally consistent with our internal thesis (leadership in accelerators + software moat).
- But portfolio risk is concentrated: Both PF002 and PF003 have significant single‑name exposure, which amplifies event risk into a high‑expectation earnings print and ongoing geopolitical/competitive risks (per internal risk assessment).

Recommended adjustments
- PF002 (Aggressive profile, very concentrated):
  - Trim 10–20% of the NVDA position pre‑earnings to reduce single‑name risk while keeping a core overweight. For 12,000 shares outstanding, that’s a 1,200–2,400 share trim. Reallocate proceeds to:
    - AI supply chain beneficiaries (e.g., TSM, AVGO) already in the book to maintain the AI theme, and/or
    - Cash buffer (5–10%) as per internal hedging guidance (From internal research).
  - Optional hedge: Layer a 1–2 month NVDA put spread (or an index hedge as per our internal memo’s QQQ put‑spread guidance) into the earnings event window to mitigate downside gap risk (From internal research).

- PF003 (Moderate profile, high single‑name weight):
  - Target a 20–25% NVDA weight over the next 4–8 weeks. Stagger trims around the earnings event (e.g., 1/3 pre‑print, 2/3 post‑print depending on results/guide). From 8,000 shares, that likely implies a 1,500–2,000 share reduction initially, with follow‑ups as needed.
  - Rebalance proceeds toward diversified AI beneficiaries (MSFT/ORCL/TSM) or maintain partial cash to redeploy on volatility.

What would change our stance
- More bullish: Evidence of accelerated Blackwell shipments, continued mid‑70s gross margins, and hyperscalers stepping up 2026 capex more than expected would support maintaining higher weights [Nasdaq](https://www.nasdaq.com/articles/ai-bubble-worries-are-rising-nvidias-319-billion-profit-says-otherwise), [Yahoo! Finance](https://finance.yahoo.com/news/ai-leader-nvidia-forecasts-fourth-212147945.html).
- More cautious: Signs of Blackwell ramp delays, margin pressure from pricing/competitive intensity, unexpected hyperscaler capex pullbacks, or tighter China controls would argue for deeper trims [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-4-trillion-stock-rally-104747765.html), [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-breaks-expectations-massive-earnings-114938956.html).

Attribution
- Competitive position and risk framework: According to internal research.
- Portfolio exposures and weights: From internal database.
- Market developments and catalysts: [Yahoo! Finance](https://finance.yahoo.com/news/ai-leader-nvidia-forecasts-fourth-212147945.html), [Nasdaq](https://www.nasdaq.com/articles/buy-3-ai-semiconductor-powerhouses-poised-dominate-2026), [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-breaks-expectations-massive-earnings-114938956.html), [Morningstar](https://www.morningstar.com/news/marketwatch/20251120244/these-were-the-five-biggest-highlights-from-nvidias-earnings), [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-4-trillion-stock-rally-104747765.html), [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-eyes-openai-investment-buy-182748007.html).

Note on gaps: We did not refresh detailed supply‑chain metrics (CoWoS/HBM allocations) or a full competitive teardown in this pass; recommendations above rely on internal thesis plus high‑level external updates. If we need a deeper dive on HBM/CoWoS capacity by vendor and AMD MI3xx share by hyperscaler, we can scope that as a separate research task.

### Query 2: Competitive Analysis (Escalation Expected)

Deep competitive analysis requires external research agent.

In [14]:
query = """
Perform a competitive analysis of AMD vs NVIDIA in the AI accelerator market:

1. First check: Do we have any internal research comparing these companies?
2. Then analyze: Current Market share trends, product roadmaps, and customer wins
3. Conclude: Which company is better positioned for 2025-2026?

Cite specific sources for your analysis.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# # display_citations(result)

🔧 internal_search_research: {
  "query": "AMD vs NVIDIA in AI accelerator/data center GPU market: market share, MI300/MI325/MI350 vs H100/H200/GB200 Blackwell, CUDA vs ROCm ecosy...
🔧 bigdata_research_agent: {
  "query": "Provide a current (through early 2026) competitive analysis of AMD vs NVIDIA in the AI accelerator / data center GPU market with inline ...


Here’s a structured competitive analysis of AMD vs. NVIDIA in AI accelerators, following your steps.

First check — internal research
- NVIDIA (internal thesis, Dec 15, 2024): Data Center revenue inflected sharply ($18.4B, +409% YoY), with Blackwell next-gen GPUs expected to deliver step-function performance; CUDA ecosystem cited at 4M+ developers creating meaningful switching costs; risks include export restrictions to China and supply tightness. According to internal research.
- AMD (internal thesis, Dec 11, 2024): MI300X highlighted for strong LLM inference with 192GB HBM3 (larger memory than H100 of that period); confirmed deployments at Microsoft Azure and Oracle Cloud; key challenge remains the ROCm software ecosystem lagging CUDA; management targeting sizable AI revenue ramp into 2025. According to internal research.
- Portfolio Strategy Update (Jan 5, 2025): Tilt increased to NVIDIA (+3% weight) and trimmed AMD (-1%) on valuation and execution risk; thesis emphasized near-term demand strength for NVIDIA AI training. According to internal research.
- Tech Risk Assessment (Jan 10, 2025): Flagged valuation risk, potential “AI bubble” dynamics, and policy risks; noted NVIDIA’s China exposure and supply-demand imbalances. According to internal research.

Analysis — market share, roadmap, customer wins (with sources)
1) Market share and trends (2023–2026)
- NVIDIA has maintained a dominant share of the data center GPU/AI accelerator market, generally cited in the mid-80s to low-90s percent range across 2024–2025 (e.g., ~92% in 2024; ~85% in 2025). See coverage summarizing third-party estimates in Nasdaq and analysis pieces: Nasdaq (92% in 2024), Nasdaq (85% in 2025), Nasdaq (85% thesis), Forbes (80–95% range 2023–2025) — [Nasdaq](https://www.nasdaq.com/articles/ai-infrastructure-could-triple-14-trillion-2030-heres-best-stock-buy-2026), [Nasdaq](https://www.nasdaq.com/articles/2-ai-stocks-buy-they-soar-20-trillion-according-wall-street-experts), [Nasdaq](https://www.nasdaq.com/articles/nvidias-85-gpu-market-share-faces-growing-competition-ai-stock-still-buy-2026), [Forbes](https://www.forbes.com/sites/roomykhan/2025/11/03/all-roads-lead-to-nvidia-bankrolling-its-own-ai-gold-rush/).
- AMD remains the primary challenger with single-digit market share through 2025, and growing momentum into 2026 from MI300 deployments and subsequent roadmap parts. See market-share framing in the above coverage — [Nasdaq](https://www.nasdaq.com/articles/2-ai-stocks-buy-they-soar-20-trillion-according-wall-street-experts), [Nasdaq](https://www.nasdaq.com/articles/nvidias-85-gpu-market-share-faces-growing-competition-ai-stock-still-buy-2026).
- Custom silicon (Google TPU, AWS Trainium/Inferentia, etc.) is scaling and increasingly contributes to deployments alongside merchant GPUs, per Dell’Oro commentary on accelerators powering DC IT growth — [Yahoo! Finance](https://finance.yahoo.com/news/ai-accelerators-power-data-center-140000960.html).
- Broader AI infrastructure spend is still expanding into 2026, reinforcing a “giga cycle” backdrop — [Yahoo! Finance](https://finance.yahoo.com/news/semiconductor-industry-enters-unprecedented-giga-165934463.html), [Yahoo! Finance](https://finance.yahoo.com/news/trends-strategies-shaping-372-6-145500659.html).

2) Product roadmaps and timelines
- NVIDIA
  - Hopper (H100/H200): H200 has remained relevant as a transitional product; reports indicate H200 shipments to China in early 2026 under evolving export permissions — [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-plans-h200-chip-shipments-174942355.html), [Yahoo! Finance](https://finance.yahoo.com/news/china-expected-approve-h200-imports-132748643.html), [Yahoo! Finance](https://finance.yahoo.com/news/white-house-decision-really-means-043200869.html).
  - Blackwell ramp (B200/GB200 and beyond): Press and industry commentary through late 2025–early 2026 point to a rapid production ramp and strong demand — [Charlotte Observer](https://www.charlotteobserver.com/news/business/article314365118.html#storylink=partnerdigest_the). OEM availability is visible; for example, Supermicro announced HGX B300 (Blackwell Ultra) liquid-cooled systems ready for high-volume shipment — [AP/PR Newswire](https://apnews.com/press-release/pr-newswire/supermicro-expands-nvidia-blackwell-portfolio-with-new-4u-and-2-ou-ocp-liquid-cooled-nvidia-hgx-b300-solutions-ready-for-high-volume-shipment-7cd7c7be5fb2299acac4cd7fb80456e3), [FinanzNachrichten](https://www.finanznachrichten.de/nachrichten-2025-12/67190180-super-micro-computer-inc-supermicro-expands-nvidia-blackwell-portfolio-with-new-4u-and-2-ou-ocp-liquid-cooled-nvidia-hgx-b300-solutions-ready-for-008.htm).
  - Performance claims: Blackwell Ultra (GB300) showed MLPerf Training leadership (e.g., Llama 3.1 405B pretrain completed 2x faster vs. GB200 and ~4x vs. H100-class systems) — [HPCwire](https://www.hpcwire.com/2025/11/12/nvidia-showcases-blackwell-ultra-performance-on-mlperf-benchmark/). NVIDIA also highlighted large efficiency gains (e.g., 10x throughput per megawatt vs. Hopper) — [Nasdaq](https://www.nasdaq.com/articles/nvidia-earnings-good-or-bad), and NVL72 system-level inference speedups vs. H100-class systems — [Nasdaq](https://www.nasdaq.com/articles/prediction-nvidia-stock-going-soar-over-next-12-months).
- AMD
  - MI300 series (MI300X/MI300A) established a beachhead with strong inference characteristics; internal work pointed to Azure and OCI availability. According to internal research.
  - Forward roadmap: AMD has publicly discussed successive Instinct generations (MI3xx to MI4xx cadence), but the most concrete, widely cited external data points in early 2026 press focus on MI300 traction; detailed MI350/MI4xx specs and timing claims are less consistently sourced in the material above. According to internal research.

3) Software ecosystem
- NVIDIA’s CUDA remains the de facto standard with deep framework/toolchain integration and is a core reason customers perceive lower TCO at the system level — [Nasdaq](https://www.nasdaq.com/articles/2-ai-stocks-buy-they-soar-20-trillion-according-wall-street-experts). Internal research also emphasizes CUDA’s developer scale and maturity. According to internal research.
- AMD’s ROCm has improved but still trails CUDA in breadth and maturity; this remains a key execution item for AMD as it scales deployments. According to internal research.

4) Customer wins and availability
- Hyperscalers/cloud:
  - AMD: Internal research confirms Microsoft Azure and Oracle Cloud deployments for MI300X; Meta adoption for inference also cited internally. According to internal research.
  - NVIDIA: Widespread availability is visible through OEM/server partners and cloud instances; the Blackwell ramp and OEM systems availability (e.g., Supermicro HGX B300 platforms) highlight readiness for large-scale deployments — [AP/PR Newswire](https://apnews.com/press-release/pr-newswire/supermicro-expands-nvidia-blackwell-portfolio-with-new-4u-and-2-ou-ocp-liquid-cooled-nvidia-hgx-b300-solutions-ready-for-high-volume-shipment-7cd7c7be5fb2299acac4cd7fb80456e3), [FinanzNachrichten](https://www.finanznachrichten.de/nachrichten-2025-12/67190180-super-micro-computer-inc-supermicro-expands-nvidia-blackwell-portfolio-with-new-4u-and-2-ou-ocp-liquid-cooled-nvidia-hgx-b300-solutions-ready-for-008.htm).
- OEMs: Supermicro’s Blackwell Ultra systems show the ecosystem momentum around NVIDIA’s platform — [AP/PR Newswire](https://apnews.com/press-release/pr-newswire/supermicro-expands-nvidia-blackwell-portfolio-with-new-4u-and-2-ou-ocp-liquid-cooled-nvidia-hgx-b300-solutions-ready-for-high-volume-shipment-7cd7c7be5fb2299acac4cd7fb80456e3).

5) Supply chain and constraints
- Policy/export: Evolving U.S.–China export controls continue to shape mix and availability; reports point to H200 being permissibly shipped to China in early 2026 while next-gen Blackwell remains restricted — [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-plans-h200-chip-shipments-174942355.html), [Yahoo! Finance](https://finance.yahoo.com/news/china-expected-approve-h200-imports-132748643.html), [Yahoo! Finance](https://finance.yahoo.com/news/white-house-decision-really-means-043200869.html).
- Internal risk work highlighted broader supply/tightness risks (including packaging/memory capacity) and elevated valuation sensitivity across AI leaders. According to internal research.

6) Performance data points
- NVIDIA: Blackwell Ultra GB300 MLPerf Training leadership (Llama3.1-405B pretraining) — [HPCwire](https://www.hpcwire.com/2025/11/12/nvidia-showcases-blackwell-ultra-performance-on-mlperf-benchmark/); system-level inference uplift vs. H100-class in NVL72 — [Nasdaq](https://www.nasdaq.com/articles/prediction-nvidia-stock-going-soar-over-next-12-months); efficiency gains vs. Hopper — [Nasdaq](https://www.nasdaq.com/articles/nvidia-earnings-good-or-bad).
- AMD: Internal research points to strong MI300 inference characteristics and competitive memory capacity, with customer deployments at Azure/OCI. According to internal research.

7) Pricing/ASP and TCO
- NVIDIA: Market commentary frequently frames NVIDIA as offering the lowest TCO at a system/platform level because of end-to-end integration (CUDA, networking, system software), despite higher component pricing — [Nasdaq](https://www.nasdaq.com/articles/2-ai-stocks-buy-they-soar-20-trillion-according-wall-street-experts). Rough GB200 cost context when deployed in NVL72 systems has been discussed — [Nasdaq](https://www.nasdaq.com/articles/prediction-nvidia-stock-going-soar-over-next-12-months).
- AMD: Internal work and customer conversations often position AMD as a cost-effective alternative — with the caveat that software maturity and integration drive real-world TCO. According to internal research.

8) Key risks/constraints
- Export controls and geographic mix: Restrictions affect product availability/mix (e.g., Hopper-era parts vs. Blackwell in China) — [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-plans-h200-chip-shipments-174942355.html), [Yahoo! Finance](https://finance.yahoo.com/news/china-expected-approve-h200-imports-132748643.html), [Yahoo! Finance](https://finance.yahoo.com/news/white-house-decision-really-means-043200869.html).
- Custom silicon from hyperscalers (TPU, Trainium/Inferentia, etc.) is growing and can displace merchant GPUs in specific workloads or tenants — [Yahoo! Finance](https://finance.yahoo.com/news/ai-accelerators-power-data-center-140000960.html).
- Valuation/execution: Internal risk assessment highlights the possibility of AI spend front-running ROI, leading to sensitivity if monetization slows, and notes NVIDIA’s relative China exposure. According to internal research.

Conclusion — who is better positioned for 2025–2026?
- NVIDIA: Better positioned overall for 2025–2026. It retains overwhelming market share and mindshare; the Blackwell ramp is underway with visible OEM/system support and benchmark leadership (MLPerf and system-level efficiency/performance), and customers often perceive lower TCO due to CUDA/software maturity and platform integration — [Nasdaq](https://www.nasdaq.com/articles/ai-infrastructure-could-triple-14-trillion-2030-heres-best-stock-buy-2026), [Nasdaq](https://www.nasdaq.com/articles/2-ai-stocks-buy-they-soar-20-trillion-according-wall-street-experts), [HPCwire](https://www.hpcwire.com/2025/11/12/nvidia-showcases-blackwell-ultra-performance-on-mlperf-benchmark/), [Nasdaq](https://www.nasdaq.com/articles/prediction-nvidia-stock-going-soar-over-next-12-months), [AP/PR Newswire](https://apnews.com/press-release/pr-newswire/supermicro-expands-nvidia-blackwell-portfolio-with-new-4u-and-2-ou-ocp-liquid-cooled-nvidia-hgx-b300-solutions-ready-for-high-volume-shipment-7cd7c7be5fb2299acac4cd7fb80456e3).
- AMD: Well-positioned to gain share as a cost-effective alternative, with MI300 deployments in major clouds and a continuing roadmap; however, AMD’s biggest swing factor remains software (ROCm ecosystem breadth/maturity) and execution at hyperscale scale-out. According to internal research.

Key swing factors to watch
- Software portability and maturity: Any material closing of the CUDA–ROCm gap could accelerate AMD share gains. According to internal research.
- Supply/policy: Export-control policy, and the availability of packaging/memory capacity, can shift shipment mix and timing — [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-plans-h200-chip-shipments-174942355.html), [Yahoo! Finance](https://finance.yahoo.com/news/china-expected-approve-h200-imports-132748643.html).
- Custom silicon adoption: Faster uptake of TPUs/Trainium/MTIA in inference/training could cap merchant GPU share expansion — [Yahoo! Finance](https://finance.yahoo.com/news/ai-accelerators-power-data-center-140000960.html).
- Economics/TCO: If customers quantify materially lower TCO with NVIDIA’s integrated stack, NVIDIA’s lead could persist; conversely, if AMD proves better cost-performance at scale with adequate software support, share could shift — [Nasdaq](https://www.nasdaq.com/articles/2-ai-stocks-buy-they-soar-20-trillion-according-wall-street-experts).

### Query 3: Internal Holdings Check (No Escalation Expected)

Simple portfolio question - should use only internal tools.

In [21]:
query = """
What is our total exposure to NVIDIA across all portfolios? 
Include position sizes, average cost basis, and unrealized P&L.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

🔧 internal_query_database: {
  "sql_query": "SELECT h.portfolio_id, p.portfolio_name, h.ticker, h.company_name, h.shares, h.avg_cost, h.current_price, h.market_value, h.unrealiz...


From internal database (as of latest internal pricing: NVDA $875.50):

By portfolio
- PF002 – AI & Semiconductor Focus
  - Position: 12,000 shares
  - Avg cost: $450.00
  - Market value: $10,506,000
  - Unrealized P&L: $5,106,000
- PF003 – Diversified Tech Leaders
  - Position: 8,000 shares
  - Avg cost: $520.00
  - Market value: $7,004,000
  - Unrealized P&L: $2,844,000
- PF001 – US Large Cap Growth
  - No NVDA position

Totals across all portfolios
- Total shares: 20,000
- Weighted avg cost: $478.00
- Total market value: $17,510,000
- Total unrealized P&L: $7,950,000

---

# 💳 Credit Research Use Cases

Questions relevant to fixed income and credit analysts.

### Query 4: Sector-Wide Credit Trends (Escalation Expected)

Macro credit analysis requires deep research.

In [23]:
query = """What are the credit risks for the semiconductor sector based on the latest news? Summarize in a 5 bullet points"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

🔧 internal_search_research: {
  "query": "semiconductor sector credit risks latest news, export controls, AI demand concentration, memory pricing cycle, CHIPS Act subsidy risks, ...
🔧 bigdata_research_agent: {
  "query": "Provide a concise, credit-focused summary of the latest risks facing the global semiconductor sector (last 30\u201360 days). Output exac...


- AI demand is highly concentrated in a few hyperscalers; outsized capex plans (e.g., Meta’s $115–$135B in 2026) boost near-term orders but increase risk to cash flow visibility if monetization lags or spending is deferred [Yahoo! Finance](https://finance.yahoo.com/news/big-tech-pushes-ai-spend-154706220.html).
- Export controls continue to cloud revenue mix and margins: conditional approvals for some advanced AI chips to China keep sales partially open but uncertain, while ongoing restrictions increase compliance risk and potential demand displacement [Associated Press](https://apnews.com/article/nvidia-trump-china-ai-a34e9e21bdc132f32cc9a448f3026da4).
- Supply-chain fragility persists: a late-December Taiwan earthquake triggered precautionary fab shutdowns, underscoring ongoing exposure to natural-disaster and operational interruption risk across leading foundry nodes [DigiTimes](https://www.digitimes.com/news/a20251229VL210/taiwan-earthquake-recovery-tsmc-production.html).
- Execution and funding risks on capacity builds: delays, cost overruns, and shifting timelines tied to major U.S. projects and incentive programs raise the chance of higher leverage needs and weaker project IRRs if subsidies come slower than spend [MSN](https://www.msn.com/en-us/money/other/america-s-biggest-chip-gamble-breaks-ground-49-000-jobs-promised-but-first-chips-won-t-roll-until-2030/ss-AA1UIZ7V?ocid=finance-verthp-feeds), [AOL](https://www.aol.com/lifestyle/u-chips-act-takes-another-130002966.html).
- Memory cycle volatility remains a swing factor: sharp DRAM/NAND price hikes forecast near term support cash generation for producers but risk affordability for OEMs and could reverse quickly if demand normalizes, pressuring free cash flow and working capital [Yahoo! Finance](https://finance.yahoo.com/news/memory-makers-prioritize-server-applications-130000793.html), [AOL](https://www.aol.com/news/shortage-memory-chips-hitting-tech-050142941.html).

### Query 5: ESG/Regulatory Risk (Escalation Expected)

In [24]:
query = """
Assess regulatory and ESG risks for our technology holdings based on latest news. 
Focus on antitrust, data privacy, and AI governance regulations.
Provide a 2 bullet point summary for each company.
"""
display_query(query)
## Uncomment to run
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)


🔧 internal_portfolio_summary: {
  "portfolio_id": "PF001"
}...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF002"
}...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF003"
}...
🔧 internal_search_research: {
  "query": "Latest regulatory and ESG risks for our tech holdings (MSFT, AAPL, META, AMZN, GOOGL, NVDA, AVGO, PLTR, AMD, TSMC, CRM, ORCL) focusing o...
🔧 bigdata_research_agent: {
  "query": "Provide a concise, up-to-date risk brief (latest developments through Jan 30, 2026) for each of these technology companies we hold: Micr...


I pulled our current holdings from internal portfolios PF001, PF002, PF003 and assessed regulatory/ESG risks focused on antitrust, data privacy, and AI governance. Below are two concise bullets per company.

Microsoft (MSFT)
- Antitrust: UK class action alleges overcharging for Windows Server licenses on rival clouds and spotlights UK/EU scrutiny of Microsoft’s cloud licensing and bundling practices, which could drive pricing/contract changes and potential remedies [US News & World Report](https://money.usnews.com/investing/news/articles/2025-12-11/microsoft-fights-2-8-billion-uk-lawsuit-over-cloud-computing-licences).
- Data privacy: EU regulators flagged GDPR issues with Microsoft 365 Education tracking students and are assessing a complaint over alleged storing of Israeli surveillance data on EU servers—raising fines/remediation risk and customer trust implications [TelecomLead](https://telecomlead.com/smart-phone/austria-rules-microsoft-365-education-tracked-students-exposing-gdpr-failures-in-schools-124262), [The Times of Israel](https://www.timesofisrael.com/microsoft-slapped-with-complaint-over-alleged-storage-of-israeli-surveillance-data/).

Apple (AAPL)
- Antitrust/DMA: EU enforcement under the DMA (anti‑steering) has already led to fines and continued compliance pressure; broader global actions (e.g., U.S., UK) keep App Store and ecosystem practices under scrutiny [Wccftech](https://wccftech.com/apple-flays-the-eus-lethargy-as-an-alternate-app-store-collapses/), [US News & World Report](https://money.usnews.com/investing/news/articles/2025-11-28/apple-tells-eu-commission-it-meets-digital-markets-act-dma-thresholds).
- Data privacy: Italy’s competition regulator fined Apple over App Tracking Transparency implementation, intensifying privacy/ads policy risk and potential changes to consent flows and ad targeting economics [Morningstar](https://www.morningstar.com/news/dow-jones/202512221384/apple-fined-115-million-in-italy-over-app-tracking-policy).

Meta (META)
- Antitrust: Ongoing U.S. and EU scrutiny of platform power, acquisitions, and interoperability could constrain product integrations and monetization in messaging/social ecosystems (According to internal research).
- Data privacy/AI governance: Heightened DSA/GDPR enforcement on ads transparency, children’s data, and AI content moderation raises compliance cost and fine risk; EU AI Act will add model/data transparency obligations (According to internal research).

Amazon (AMZN)
- Antitrust: FTC monopolization case targets marketplace practices and seller treatment; potential structural or behavioral remedies could affect fees, Buy Box, and Prime economics (trial timeline into 2026) (According to internal research).
- Data privacy/AI governance: DSA VLOP duties (content moderation, ad transparency) and evolving AI safety expectations for Alexa/GenAI services increase compliance and reputational risk (According to internal research).

Alphabet/Google (GOOGL)
- Antitrust/DMA: EU DMA compliance proceedings and U.S. DOJ cases (Search, AdTech) point to potential remedies impacting default placements, data access, and ad tech stack integration (According to internal research).
- Data privacy/AI governance: Settlements and ongoing suits over data collection/use, plus EU AI Act and DSA obligations (risk assessments, transparency) raise costs and constrain how user and publisher data feed AI models (According to internal research).

NVIDIA (NVDA)
- Export controls/market power: Tightened U.S. restrictions on advanced AI GPUs to China and potential scrutiny of supply/pricing practices could affect revenue mix and unit allocation (According to internal research).
- AI governance/IP: Heightened scrutiny on training data provenance and model responsibility for enterprise AI software stacks may add contractual, indemnity, and compliance burdens (According to internal research).

Broadcom (AVGO)
- Antitrust/post‑merger conduct: Customer and industry complaints about VMware licensing changes post‑acquisition elevate EU/UK attention; potential commitments could limit pricing/bundling flexibility (According to internal research).
- Contractual/ESG: Perceived “lock‑in” and unilateral term changes risk litigation and reputational damage with large enterprise/public‑sector clients (According to internal research).

Palantir (PLTR)
- Data privacy/ethics: Government surveillance contracts (immigration, health, law enforcement) drive sustained NGO/regulatory scrutiny over proportionality, consent, and data minimization (GDPR/FOI challenges) (According to internal research).
- AI governance: EU AI Act “high‑risk” categories (public sector, biometrics, safety‑critical) imply documentation, human oversight, and transparency obligations affecting deployment velocity and margins (According to internal research).

AMD (AMD)
- Export controls: U.S. curbs on advanced AI accelerators to China impact product roadmaps and volumes; China‑specific SKUs face approval uncertainty and pricing pressure (According to internal research).
- AI governance/IP: Expanding accelerator/software ecosystem raises exposure to data/IP disputes and enterprise AI responsibility requirements (model cards, transparency, security-by-design) (According to internal research).

TSMC (TSM)
- Geopolitical/export controls: U.S.–China tensions and evolving export rules constrain leading‑edge tool/technology flows and customer mix; reshoring and supply‑chain redundancy add cost (According to internal research).
- Data security/IP: Ongoing trade‑secrets/IP‑theft risks across the supply chain require elevated security/compliance investment and can trigger legal exposure (According to internal research).

Salesforce (CRM)
- Data privacy/security: Third‑party and integration‑layer exposures elevate incident risk; contractual/data‑processing addenda and sectoral rules (HIPAA/financial) drive compliance spend (According to internal research).
- AI governance: EU AI Act timelines (first obligations in 2026) will require risk management, documentation, and transparency for CRM AI/assistants, affecting time‑to‑market and support costs (According to internal research).

Oracle (ORCL)
- Data privacy/residency: Growth in regulated‑sector cloud and “sovereign” deployments elevates obligations around localization, access controls, and auditability, with contractual risk allocation under scrutiny (According to internal research).
- AI governance/capex: Building AI data center capacity carries policy risk (energy, permits) and EU AI Act compliance duties; potential oversupply or regulatory delays could pressure returns (According to internal research)

---

## 📊 Observability

View detailed traces in LangSmith:
- Each query shows the **tool call sequence**
- See which tools were checked first vs escalated
- Monitor **latency** differences between internal vs external calls
- Track **token usage** for cost optimization

**Dashboard:** https://smith.langchain.com


## 🔟 Key Benefits of This Architecture

1. **Internal-first** — Fast answers from DB and vector store; escalate only when external research is needed.
2. **Full citations** — Research Agent returns inline [1], [2] and numbered sources; display via `display_citations()`.
3. **Production-ready** — Research client has retry and stream timeout.
4. **Reusability** — `create_financial_database()`, `create_vector_store()`, and tools come from `langgraph_core.py`.
5. **Observability** — LangSmith traces show when the agent uses internal vs Research Agent tools.

## 🎯 Next Steps

- **Tune escalation** — Adjust system prompt so the agent escalates only for “deep research” or “recent market” questions.
- **Research effort** — Use `research_effort="lite"` for faster, lighter answers or `"standard"` for full depth.
- **Follow-up** — Use `result.chat_id` and `client.follow_up()` for multi-turn Research Agent conversations.
- **LangSmith** — Use traces to see tool order and Research Agent usage.

---

## 📚 Additional Resources

- **Bigdata.com API docs**: https://docs.bigdata.com
- **Research Agent client**: `Research_Agent_Sync_Response/README.md` (retry, stream timeout)
- **LangGraph / LangChain**: https://langchain-ai.github.io/langgraph/
- **LangSmith**: https://smith.langchain.com
- **Agent_To_BigData README**: `README.md` in this folder

**Questions?** support@bigdata.com